# Session 10 — DQN lab, an RL-based agent

**DAT 5668 · Module 5 · Practice session (about 80 minutes)**

**Family this session: RL-based agent.** Compare the learned policy with a base-stock rule. Do not describe this network as an LLM-based agent.

![Deep Q-network](../../assets/dqn.png)

## Run of show
- 10 min — Baseline first. A neural net that loses to "order up to 2" is not a win.
- 55 min — Finish the TD target and the weight update. Compare DQN to the base-stock rule.
- 15 min — Deploy / do not deploy, and the human override.

Instructional inventory again. State features are scaled stock and the demand just observed. Actions are still order 0, 1, or 2.


In [ ]:
import numpy as np

CAPACITY = 4
GAMMA = 0.95
HOLD, MARGIN, STOCKOUT = 0.2, 2.0, 1.5
RNG = np.random.default_rng(11)


def step(stock, order):
    demand = int(RNG.integers(0, 3))
    started = min(CAPACITY, stock + order)
    sales = min(started, demand)
    ending = started - sales
    missed = max(0, demand - started)
    pay = MARGIN * sales - HOLD * ending - STOCKOUT * missed
    return ending, pay, demand


def features(stock, demand):
    return np.array([stock / CAPACITY, demand / 2.0])


class TinyQ:
    def __init__(self, rng):
        self.w1 = rng.normal(0, 0.2, size=(2, 8))
        self.b1 = np.zeros(8)
        self.w2 = rng.normal(0, 0.2, size=(8, 3))
        self.b2 = np.zeros(3)

    def forward(self, x):
        self.x = np.asarray(x, dtype=float)
        self.z1 = self.x @ self.w1 + self.b1
        self.h = np.maximum(0, self.z1)
        return self.h @ self.w2 + self.b2

    def learn(self, x, action, target, lr=0.05):
        """TODO: move w2/b2 (and, if you have time, w1/b1) so Q(x, action) gets closer to target.

        Hint: q = forward(x). You only want to change the output for `action`.
        grad_q = q.copy(); grad_q[action] = q[action] - target
        Then dw2 = outer(h, grad_q); subtract lr * dw2 from w2.
        """
        q = self.forward(x)
        # TODO
        return q


def base_stock_return(days=300):
    stock, demand, total = 1, 1, 0.0
    for _ in range(days):
        order = max(0, min(2, 2 - stock))
        stock, pay, demand = step(stock, order)
        total += pay
    return total


print("base-stock return over 300 days", round(base_stock_return(), 2))


In [ ]:
def train(steps=1500, epsilon=0.20):
    online = TinyQ(np.random.default_rng(3))
    memory = []
    stock, demand = 1, 1
    recent = []
    for t in range(steps):
        x = features(stock, demand)
        q = online.forward(x)
        action = int(RNG.integers(0, 3)) if RNG.random() < epsilon else int(np.argmax(q))
        ending, pay, new_demand = step(stock, action)
        memory.append((x, action, pay, features(ending, new_demand)))
        # TODO: sample one memory tuple and call online.learn with the TD target
        # gamma * max(online.forward(next_x)) + pay.
        # Using the online net as the target is acceptable for this lab; say in the debrief why a frozen copy is safer.
        recent.append(pay)
        stock, demand = ending, new_demand
    return online, float(np.mean(recent[-100:]))


net, avg_pay = train()
print("average pay of last 100 steps", round(avg_pay, 3))
print("Fill the TODO. Then compare this average with the base-stock cell before you recommend deployment.")


## Written close (in class)

1. Did DQN beat, match, or lose to base-stock on your run?
2. One failure you would watch in a longer run (divergence, no exploration, or a reward that ignores stockouts).
3. The human override you would require if this touched a real purchase order.


## Fill this with me

Before you trust the printout, bet as a class. Then point at the two numbers and say whether you would sign a purchase order.


In [ ]:
# FILL WITH THE CLASS. "yes" or "no" — will this short run beat base-stock?
beats_base_stock = ""
# FILL WITH THE CLASS after you see the numbers. "deploy" or "do not deploy".
our_call = ""
print("Bet:", beats_base_stock or "(bet first)")
print("Call:", our_call or "(after the numbers)")
